In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.7 The Kohn–Sham Construction

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.7",
    title="The Kohn–Sham Construction",
    blurb="The most-used method in computational science, built end to end: orbitals "
    "restore the kinetic energy Thomas–Fermi butchered, the uniform gas of 8.4 "
    "supplies the exchange-correlation input, and a working radial code solves "
    "helium, beryllium, and neon to the reference values. Then the honest "
    "accounting: the hydrogen atom's self-interaction disaster, a local-density "
    "approximation built from scratch for the exact laboratory's own interaction "
    "and judged against the exact answer, and the face-off between the LDA "
    "potential and the exact one recovered in 8.6.",
    difficulty="advanced",
    estimate="140–170 min",
)

## Notebook overview

This is the notebook the movement was built toward. Thomas–Fermi
([§8.5](thomas-fermi.ipynb)) showed a density-only theory is *possible* and its
local kinetic energy is fatal; Hohenberg–Kohn ([§8.6](hohenberg-kohn.ipynb))
proved the density suffices and even computed the exact Kohn–Sham potential of
the laboratory. Kohn and Sham's 1965 move {cite}`kohnsham1965` assembled the
pieces into the scheme that now consumes a measurable share of the world's
supercomputing: *keep orbitals for the kinetic energy* (the fictitious
non-interacting system of the [§8.6](hohenberg-kohn.ipynb) inversion, now made
the engine rather than the diagnostic), and shovel everything unknown into an
exchange-correlation functional $E_{xc}[n]$ — which the homogeneous electron gas
of [§8.4](hartree-fock-gas.ipynb) supplies in its local-density form.

The program: derive the KS equations and assemble the LDA potential from the
[§8.4](hartree-fock-gas.ipynb) deliverable (certifying the correlation
potential's derivative by central differences, in the
[§8.5](thomas-fermi.ipynb) toolkit's spirit); study the self-consistent loop as
the damped fixed point it is; solve He, Be, and Ne against the standard
reference values (shells restored, $2p$ channel and all); then the honest
ledger, in three exhibits. The hydrogen atom, where one electron interacts with
itself through the mean field and LDA misses the exact $-1/2$ Ha by $11\%$
while its eigenvalue misses by *half*. A local-density approximation *built
from scratch* for the laboratory's own soft interaction — its uniform-gas
exchange computed by two independent routes that agree to better than $10^{-4}$ — and
judged against the exact answer the laboratory knows. And the face-off: the
LDA exchange-correlation potential laid over the exact one recovered in
[§8.6](hohenberg-kohn.ipynb), where the local approximation's exponentially
dying tail against the exact $-1/|x|$ makes the band-gap troubles of
[§8.8](exact-conditions-band-gap.ipynb) visible a notebook early.

> **Conventions (this notebook).** Hartree atomic units. Three-dimensional
> atoms are spin-unpolarized (all systems here are closed-shell except
> hydrogen, whose unpolarized treatment is deliberate and discussed), on the
> uniform radial grid $r_j = jh$, $h = 25/8000$ Bohr, with the Hartree
> potential by the shell theorem's two cumulative integrals (`numpy.cumsum`,
> an $O(N)$ replacement for the dense kernel) and eigenproblems per angular
> channel by `scipy.linalg.eigh_tridiagonal` (the $l(l+1)/2r^2$ centrifugal
> term added for $p$ states). The LDA uses [§8.4](hartree-fock-gas.ipynb)
> exchange and Perdew–Zunger correlation {cite}`perdewzunger1981`. Total
> energies use the standard double-counting-corrected assembly
> $E = \sum_i f_i\varepsilon_i - \tfrac12\!\int\! v_H n - \!\int\! v_{xc} n +
> E_{xc}[n]$. One-dimensional laboratory work reuses the
> [§8.2](exact-laboratory.ipynb) grid and interaction.
>
> **How to read the checks.** Each exercise closes with a `validate` call
> against an independent fact: a derivative identity, the standard reference
> energies, the exact laboratory, the [§8.6](hohenberg-kohn.ipynb) potential.
> A ✓ is strong evidence; a ✗ is a prompt to locate the discrepancy, not an
> automatic verdict.
>
> **Scope.** LDA only, deliberately: gradient corrections, hybrids, and the
> rest of the functional zoo are surveyed against exact constraints in
> [§8.8](exact-conditions-band-gap.ipynb), and Martin {cite}`martin2004`,
> Ch. 8–9, and Parr & Yang {cite}`parryang1989`, Ch. 7–8, carry the full
> story. The reference atomic values quoted are the standard local-density
> benchmarks (the NIST atomic reference data; tiny differences between
> correlation parametrizations are noted where they show).

## Theory in brief

### The construction

Kohn and Sham {cite}`kohnsham1965` split Levy's functional
({eq}`eq-hk-levy`) not where nature suggests but where computation does:

```{math}
:label: eq-ks-split
F[n] = T_s[n] + E_H[n] + E_{xc}[n],
\qquad
E_{xc}[n] \equiv (T - T_s) + (W - E_H),
```

with $T_s$ the non-interacting kinetic functional evaluated through orbitals
and $E_{xc}$ *defined* as everything left over — the kinetic correlation the
[§8.6](hohenberg-kohn.ipynb) Levy bound measured, plus the interaction beyond
Hartree. Varying the total energy over orbitals (the
[§8.3](hartree-fock-atoms.ipynb) constrained-variation pattern) gives
one-electron equations in a *local* potential,

```{math}
:label: eq-ks-equations
\Big[ -\tfrac12\nabla^2 + v_{\mathrm{ext}} + v_H[n] + v_{xc}[n] \Big]
\varphi_i = \varepsilon_i \varphi_i,
\qquad
v_{xc} = \frac{\delta E_{xc}}{\delta n},
\qquad
n = \sum_i f_i\,|\varphi_i|^2 ,
```

a self-consistency loop identical in shape to Hartree–Fock's but with
exchange's nonlocal matrix replaced by a function of the local density. All
the many-body difficulty now lives in one functional, and the total energy is
assembled without double counting as

```{math}
:label: eq-ks-energy
E = \sum_i f_i\,\varepsilon_i
\;-\; \tfrac12\!\int\! v_H\,n
\;-\; \!\int\! v_{xc}\,n
\;+\; E_{xc}[n]
```

(the eigenvalue sum counts the mean field once per electron; the corrections
put back the halves — the bookkeeping trap first met in
[§8.2](exact-laboratory.ipynb), now in its canonical form).

### The local-density approximation

The one functional the course has *earned* is local: assign each point the
exchange-correlation energy a uniform gas of that density would have,

```{math}
:label: eq-ks-lda
E_{xc}^{\mathrm{LDA}}[n] = \!\int\! n(\mathbf r)\,
\varepsilon_{xc}\big(n(\mathbf r)\big)\,d^3r,
\qquad
v_{xc}^{\mathrm{LDA}} = \frac{d\big(n\,\varepsilon_{xc}\big)}{dn}
= \varepsilon_{xc} + n\,\frac{d\varepsilon_{xc}}{dn},
```

with $\varepsilon_{xc} = \varepsilon_x + \varepsilon_c$ exactly the
[§8.4](hartree-fock-gas.ipynb) deliverable: $\varepsilon_x = -\tfrac34
(3/\pi)^{1/3} n^{1/3}$ (so $v_x = -(3/\pi)^{1/3} n^{1/3}$, the derivative
picking up $4/3$) and the Perdew–Zunger $\varepsilon_c(r_s)$, whose $v_c$
follows from the same rule via $r_s(n)$. Why should a uniform-gas fact serve
atoms whose density spans six orders of magnitude? Two structural reasons,
both met already: the LDA hole inherits the exact $-1$ sum rule from the real
gas it is stolen from ([§8.2](exact-laboratory.ipynb)'s constraint), and
energies integrate over the hole's *spherical average* only, forgiving much
local error. The approximation's failures are equally structural — above all
**self-interaction**: in Eq. {eq}`eq-ks-equations` each electron's own charge
sits in $v_H$, and where Hartree–Fock's $K_{ii} = J_{ii}$
([§8.3](hartree-fock-atoms.ipynb)) cancelled it exactly, the LDA can only
cancel it approximately. One electron alone in the universe still repels
itself. Hydrogen will put a number on that.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from scipy.linalg import eigh_tridiagonal

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

# --- 3-D LDA ingredients (the 8.4 deliverable, now with derivatives) ---
CX_LDA = -(3.0 / 4.0) * (3.0 / np.pi) ** (1.0 / 3.0)  # exchange energy coefficient


def eps_c_pz_vec(rs):
    """Perdew-Zunger correlation energy per electron, vectorized over r_s.

    The two-branch fit of Eq. eq-heg-pz through the Ceperley-Alder energies,
    in Hartree (section 8.4).

    Parameters
    ----------
    rs : numpy.ndarray
        Wigner-Seitz radii in Bohr.

    Returns
    -------
    numpy.ndarray
        Correlation energy per electron.
    """
    rs = np.asarray(rs, dtype=float)
    out = np.empty_like(rs)
    hi = rs >= 1.0
    out[hi] = -0.1423 / (1.0 + 1.0529 * np.sqrt(rs[hi]) + 0.3334 * rs[hi])
    lo = ~hi
    out[lo] = (
        0.0311 * np.log(rs[lo])
        - 0.048
        + 0.0020 * rs[lo] * np.log(rs[lo])
        - 0.0116 * rs[lo]
    )
    return out


def v_c_pz_vec(rs):
    """Perdew-Zunger correlation potential v_c = d(n eps_c)/dn, vectorized.

    The published closed forms of the density derivative: for rs >= 1,
    v_c = eps_c (1 + 7/6 beta1 sqrt(rs) + 4/3 beta2 rs) / (1 + beta1 sqrt(rs)
    + beta2 rs); for rs < 1 the logarithmic branch's derivative. Certified
    against a central difference of n eps_c in Exercise 1.

    Parameters
    ----------
    rs : numpy.ndarray
        Wigner-Seitz radii in Bohr.

    Returns
    -------
    numpy.ndarray
        Correlation potential in Hartree.
    """
    rs = np.asarray(rs, dtype=float)
    out = np.empty_like(rs)
    hi = rs >= 1.0
    denom = 1.0 + 1.0529 * np.sqrt(rs[hi]) + 0.3334 * rs[hi]
    out[hi] = (
        (-0.1423 / denom)
        * (1.0 + (7.0 / 6.0) * 1.0529 * np.sqrt(rs[hi]) + (4.0 / 3.0) * 0.3334 * rs[hi])
        / denom
    )
    lo = ~hi
    out[lo] = (
        0.0311 * np.log(rs[lo])
        - 0.048
        - 0.0311 / 3.0
        + (2.0 / 3.0) * 0.0020 * rs[lo] * np.log(rs[lo])
        + (2.0 * (-0.0116) - 0.0020) * rs[lo] / 3.0
    )
    return out


def rs_of_n(n):
    """Wigner-Seitz radius of a 3-D density, with a floor for empty regions.

    Parameters
    ----------
    n : numpy.ndarray
        Density in Bohr^-3.

    Returns
    -------
    numpy.ndarray
        r_s = (3/(4 pi n))^(1/3).
    """
    return (3.0 / (4.0 * np.pi * np.clip(n, 1e-30, None))) ** (1.0 / 3.0)


# --- the radial Kohn-Sham engine ---
N_R, R_MAX = 8000, 25.0
H_R = R_MAX / N_R
r = H_R * np.arange(1, N_R + 1)
OFF_R = np.full(N_R - 1, -0.5 / H_R**2)


def hartree_radial(charge_shell):
    """Radial Hartree potential by the shell theorem's two cumulative integrals.

    v_H(r) = Q_in(r)/r + integral_r^inf q(r')/r' dr' with q = 4 pi r'^2 n:
    charge inside acts from the center, shells outside contribute their
    surface value. Both integrals are running sums, O(N) via numpy.cumsum
    (the dense kernel of section 8.3 would be O(N^2) and is unnecessary for
    a local theory).

    Parameters
    ----------
    charge_shell : numpy.ndarray
        The shell charge q(r) = 4 pi r^2 n(r) on the grid.

    Returns
    -------
    numpy.ndarray
        The Hartree potential in Hartree.
    """
    # cumulative TRAPEZOID sums, not rectangles: the half-cell correction is
    # an O(h) accuracy difference worth several mHa in a total energy (the
    # rectangle version was caught by the reference-value gate).
    from scipy.integrate import cumulative_trapezoid

    q_in = (
        cumulative_trapezoid(charge_shell, r, initial=0.0) + charge_shell[0] * H_R / 2.0
    )
    over_r = charge_shell / r
    q_out_rev = cumulative_trapezoid(over_r[::-1], dx=H_R, initial=0.0)[::-1]
    return q_in / r + q_out_rev


def solve_atom_lda(Z, channels, mixing=0.35, tol=1e-9, max_iter=300):
    """Self-consistent spin-unpolarized radial Kohn-Sham LDA for one atom.

    Iterates Eq. eq-ks-equations channel by channel (the l(l+1)/2r^2
    centrifugal term added per angular momentum), mixes densities linearly,
    and assembles the total energy by the double-counting-corrected
    Eq. eq-ks-energy at every step until it settles.

    Parameters
    ----------
    Z : float
        Nuclear charge.
    channels : dict
        Mapping l -> list of occupations of that channel's lowest states,
        e.g. {0: [2, 2], 1: [6]} for neon.
    mixing : float, optional
        Linear density-mixing fraction (default 0.35).
    tol : float, optional
        Energy convergence threshold in Hartree.
    max_iter : int, optional
        Iteration cap.

    Returns
    -------
    tuple
        ``(E, eigenvalues, density, history)``: total energy, dict of orbital
        energies per l, converged density n(r), and the per-iteration |dE|.
    """
    density = None
    E_old, history = np.inf, []
    for _ in range(max_iter):
        if density is None:
            v_H = np.zeros(N_R)
            v_xc = np.zeros(N_R)
        else:
            v_H = hartree_radial(4.0 * np.pi * r**2 * density)
            nn = np.clip(density, 1e-30, None)
            rs = rs_of_n(nn)
            v_xc = (4.0 / 3.0) * CX_LDA * nn ** (1.0 / 3.0) + v_c_pz_vec(rs)
        new_density = np.zeros(N_R)
        eig_out = {}
        band_sum = 0.0
        for l, occs in channels.items():
            diag = 1.0 / H_R**2 - Z / r + v_H + v_xc + l * (l + 1) / (2.0 * r**2)
            eps_l, u_l = eigh_tridiagonal(
                diag, OFF_R, select="i", select_range=(0, len(occs) - 1)
            )
            eig_out[l] = eps_l
            for i, occ in enumerate(occs):
                u_i = u_l[:, i] / np.sqrt(H_R)
                new_density += occ * u_i**2 / (4.0 * np.pi * r**2)
                band_sum += occ * eps_l[i]
        if density is not None:
            nn = np.clip(density, 1e-30, None)
            rs = rs_of_n(nn)
            shell = 4.0 * np.pi * r**2
            e_xc_dens = CX_LDA * nn ** (4.0 / 3.0) + nn * eps_c_pz_vec(rs)
            E = (
                band_sum
                - 0.5 * np.trapezoid(v_H * density * shell, r)
                - np.trapezoid(v_xc * density * shell, r)
                + np.trapezoid(e_xc_dens * shell, r)
            )
            history.append(abs(E - E_old))
            if history[-1] < tol:
                return E, eig_out, density, np.array(history)
            E_old = E
        density = (
            new_density
            if density is None
            else (1.0 - mixing) * density + mixing * new_density
        )
    return E_old, eig_out, density, np.array(history)

## Exercise 1 — The LDA potential, assembled and certified

Equation {eq}`eq-ks-lda` turns the [§8.4](hartree-fock-gas.ipynb) energy curve
into a potential by differentiation, and transcribing published derivative
formulas is exactly the kind of step that deserves a gate: a dropped $7/6$ in
the Perdew–Zunger $v_c$ would poison every result downstream while looking
perfectly plausible. The certificate is the
[§8.5](thomas-fermi.ipynb) functional-derivative discipline applied to a plain
function: $v_{xc}(n)$ must equal the central difference of
$n\,\varepsilon_{xc}(n)$.

**Part a)** Evaluate the assembled $v_x(n) = \tfrac43 C_x n^{1/3}$ and the
Setup helper `v_c_pz_vec` on twenty densities log-spaced over
$n \in [10^{-4}, 10]$ (`numpy.geomspace`).

**Part b)** Certify both against
$\big[(n+\delta)\varepsilon_{xc}(n+\delta) - (n-\delta)\varepsilon_{xc}(n-\delta)\big]/2\delta$
with $\delta = 10^{-6}\,n$ (`numpy` arithmetic; relative step, since the
densities span five decades). Agreement at $10^{-6}$ certifies the
transcription; the ladder to every atom below rests on these twenty numbers.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the transcription is faithful

The closed-form potential must match the defining derivative to $10^{-6}$
across all twenty densities, both Perdew–Zunger branches included.

In [ ]:
validate.close(
    dev_vxc, 0.0, "v_xc equals d(n eps_xc)/dn across five decades", rtol=0.0, atol=1e-6
)
validate.check(
    bool(np.all(v_xc_claimed < 0.0)),
    "the xc potential is attractive throughout",
    "electrons dig their own holes",
)

## Exercise 2 — The loop, watched: mixing and the fixed point

The KS equations demand self-consistency, and
[§0.2](../00-foundations/root-finding.ipynb) taught the vocabulary: the SCF
cycle is a fixed-point map on densities, linear mixing
$n \leftarrow (1-\alpha)\,n + \alpha\,n_{\mathrm{new}}$ is damped iteration,
and the damping trades speed against stability. Helium is the clean test bed.

**Part a)** Run the Setup engine `solve_atom_lda` on helium at mixing
$\alpha = 0.1, 0.35, 0.7$ and record the per-iteration $|\Delta E|$ histories.

**Part b)** Plot the three convergence traces on a `matplotlib` log axis and
confirm the trade: all three reach the *same* energy to $10^{-8}$ Ha (a fixed
point does not care how it is approached), with iteration counts falling as
$\alpha$ grows — and with the standing warning that in harder systems (small
gaps, charge sloshing between distant regions) large $\alpha$ turns
oscillatory or divergent, which is why production codes ship with Anderson,
Broyden, and Pulay accelerators.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2 — the fixed point is unique

The three runs must agree to $10^{-8}$ Ha, and the iteration count must fall
monotonically with $\alpha$ on this well-behaved system.

In [ ]:
E_vals = list(energies_mix.values())
validate.close(
    max(E_vals) - min(E_vals),
    0.0,
    "one destination for every damping",
    rtol=0.0,
    atol=1e-8,
)
validate.check(
    len(traces[0.1]) > len(traces[0.35]) > len(traces[0.7]),
    "heavier damping pays in iterations",
    f"{len(traces[0.1])} > {len(traces[0.35])} > {len(traces[0.7])}",
)

## Exercise 3 — Helium, beryllium, neon: the working theory

The engine now earns its keep on three closed-shell atoms against the standard
local-density reference values (the NIST atomic benchmarks; the uniform grid
and the correlation parametrization together account for differences at a few
parts in $10^4$ of the totals): total
energies $-2.8348$, $-14.4472$, $-128.2335$ Ha, and the orbital-energy
spectra including neon's $2p$ channel — the volume's first $l > 0$ state,
carried by the centrifugal term in the Setup engine.

**Part a)** Solve He ($1s^2$), Be ($1s^2 2s^2$), and Ne ($1s^2 2s^2 2p^6$)
with `solve_atom_lda` and tabulate totals and eigenvalues against the
references (He $\varepsilon_{1s} = -0.5704$; Be $-3.8564, -0.2057$; Ne
$-30.306, -1.3228, -0.4980$ Ha).

**Part b)** Plot beryllium's radial density $4\pi r^2 n$: the two shells are
back — the structure Thomas–Fermi could not make
({numref}`fig-tf-density`) restored by exactly the orbital kinetic energy
Kohn–Sham reinstated. Alongside, tabulate the three-way comparison for He and
Be: LDA total vs the Hartree–Fock limits of
[§8.3](hartree-fock-atoms.ipynb) vs exact — LDA's *totals* are worse than
Hartree–Fock's (self-interaction inflates them), yet its energy *differences*
(the currency of chemistry) compete, which is the paradox that made DFT's
fortune.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3 — the reference values, hit

All three totals within $5\times10^{-4}$ relative of the references (grid plus
parametrization flavor), every tabulated eigenvalue within $3\times10^{-3}$,
and the LDA totals must sit *above* the Hartree–Fock limits for He and Be:
self-interaction, visible in the bottom line.

In [ ]:
for atom in ("He", "Be", "Ne"):
    validate.close(
        results[atom][0], REFS[atom]["E"], f"{atom} LDA total energy", rtol=5e-4
    )
    for (l, i), ref in REFS[atom]["eps"].items():
        validate.close(
            results[atom][1][l][i],
            ref,
            f"{atom} eigenvalue (l={l}, state {i})",
            rtol=3e-3,
        )
validate.check(
    all(results[a][0] > E_HF_LIMITS[a] for a in ("He", "Be")),
    "LDA totals sit above the HF limits (self-interaction shows)",
    "totals are the wrong scoreboard; differences are the right one",
)

## Exercise 4 — One electron, repelling itself

The cleanest indictment in density-functional theory: hydrogen. One electron,
so the exact answer is $E = -1/2$ Ha with $\varepsilon_{1s} = -1/2$, and the
exact functional would deliver it — the Hartree self-repulsion must be
*exactly* cancelled by exchange-correlation, as Hartree–Fock's
$K_{11} = J_{11}$ did in [§8.3](hartree-fock-atoms.ipynb). The LDA cannot
manage the cancellation: its $v_{xc}$, built for a sea of electrons, only
roughly offsets $v_H$ for a single one. (The calculation is spin-unpolarized,
the cleanest form of the demonstration; the spin-polarized LSD variant
softens the numbers without curing the disease.)

**Part a)** Solve hydrogen with the engine (`{0: [1]}` occupation) and report
$E$ and $\varepsilon_{1s}$ against the exact $-1/2$ and $-1/2$.

**Part b)** Plot the residual potential $v_H + v_{xc}$ (which the exact
functional would make identically zero) against $v_H$ itself: the
uncancelled self-interaction, in person. The energy misses by $11\%$; the
*eigenvalue* misses by half — and since [§8.6](hohenberg-kohn.ipynb) proved
the exact HOMO equals $-I$, the LDA eigenvalue's failure is not cosmetic but
the seed of every underestimated gap in [§8.8](exact-conditions-band-gap.ipynb).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4 — the disaster, quantified

The energy must land at the standard unpolarized-LDA value $-0.4459$ Ha (an
$11\%$ miss), the eigenvalue at $-0.2336$ Ha (a $53\%$ miss against the
ionization theorem), and the residual potential must be genuinely repulsive
where the electron lives (positive at the density's peak shell).

In [ ]:
validate.close(E_h, -0.4459, "the unpolarized-LDA hydrogen energy", rtol=1e-3)
validate.close(eig_h[0][0], -0.2336, "the LDA hydrogen eigenvalue", rtol=2e-3)
validate.check(
    residual_h[int(np.argmax(4.0 * np.pi * r**2 * dens_h))] > 0.1,
    "the uncancelled self-interaction is repulsive at the density peak",
    f"{residual_h[int(np.argmax(4.0 * np.pi * r**2 * dens_h))]:.3f} Ha",
)

## Exercise 5 — An LDA of our own: the laboratory judged exactly

Every LDA number so far leaned on the three-dimensional gas. The laboratory of
[§8.2](exact-laboratory.ipynb) deserves better: its interaction is the soft
$w(u) = 1/\sqrt{u^2+1}$, so the *honest* local approximation for it must be
built from the uniform 1-D gas with that same interaction — a functional
constructed from scratch, for our own world, and then judged against the exact
answer. The uniform-gas exchange comes out by the
[§8.4](hartree-fock-gas.ipynb) route: the one-body density matrix of the 1-D
unpolarized gas is $\rho_\sigma(u) = \sin(k_F u)/\pi u$ with
$k_F = \pi n/2$, so

```{math}
:label: eq-ks-lab-ex
\varepsilon_x(n) = -\frac{2}{n}\int_0^\infty
\left(\frac{\sin k_F u}{\pi u}\right)^{\!2} \frac{du}{\sqrt{u^2+1}},
```

with an independent cross-check in momentum space: the same quantity via the
interaction's Fourier transform $\tilde w(q) = 2K_0(|q|)$ (a modified Bessel
function, `scipy.special.k0`) integrated over two Fermi seas. Exchange only,
deliberately: correlation of the 1-D gas would need its own quantum Monte
Carlo, and the exchange-only comparison against exact and Hartree–Fock is
already the lesson.

**Part a)** Tabulate Eq. {eq}`eq-ks-lab-ex` by `scipy.integrate.quad` on 160
densities to $n = 1.6$, cross-check three of them against the $K_0$ route
(`scipy.integrate.dblquad` over the two Fermi seas, agreement below $10^{-4}$ —
the $K_0$ kernel's logarithmic point limits that route's quadrature),
and build $v_x(n)$ as the derivative of a `scipy.interpolate.CubicSpline`
through $n\,\varepsilon_x(n)$.

**Part b)** Run the 1-D Kohn–Sham loop on the laboratory (grid and $v_H$ from
the [§8.2](exact-laboratory.ipynb) machinery, density mixing $1/2$) and place
the result on the laboratory's ladder: exact $-2.2386$, Hartree–Fock
$-2.2245$, and now LDA-exchange — plus the eigenvalue against the ionization
theorem the exact functional passed at $0.3\%$ in
[§8.6](hohenberg-kohn.ipynb).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5 — the ladder's fourth rung

The two exchange routes must agree below $10^{-4}$ (the $K_0$ kernel's
logarithmic point limits the momentum-space quadrature); the LDA-exchange total must
land at its measured $-2.1584$ Ha (above Hartree–Fock — locality *loses*
exchange energy — and $3.6\%$ above exact); and the frontier eigenvalue must
miss the ionization theorem by its measured $38\%$: the hydrogen story,
recurring in a system with a known exact answer.

In [ ]:
validate.check(
    max(route_devs) < 1e-4,
    "two derivations of the gas exchange agree",
    f"max deviation {max(route_devs):.1e}",
)
validate.close(E_lab_lda, -2.15845, "the laboratory's LDA-exchange energy", rtol=1e-3)
validate.check(
    E_lab_lda > E_HF_LAB > E_EXACT_LAB,
    "the energy ladder orders correctly",
    f"{E_lab_lda:.4f} > {E_HF_LAB} > {E_EXACT_LAB}",
)
validate.close(eps_lab[0], -0.46732, "the LDA frontier eigenvalue", rtol=2e-3)
validate.check(
    abs(eps_lab[0] + I_LAB) / I_LAB > 0.3,
    "the ionization theorem fails by over 30% in LDA",
    f"{100 * abs(eps_lab[0] + I_LAB) / I_LAB:.0f}% (exact functional: 0.3%)",
)

## Exercise 6 — Face-off: the LDA potential against the exact one

[§8.6](hohenberg-kohn.ipynb) recovered the *exact* $v_{xc}$ of the laboratory
by inversion; Exercise 5 built the LDA's. Laying one over the other is the
movement's most honest picture, and it teaches a distinction every DFT
practitioner eventually internalizes: *energies forgive, potentials punish*.
The LDA exchange **energy** of the laboratory lands within ten percent of the
exact $E_x = -J/2$ (the integral averages over the hole, and the sum rule
does quiet work); the LDA **potential** — the functional's derivative, the
thing the orbitals actually feel — is off by nearly $40\%$ at the very center
and collapses in the tail, where the exact potential falls as the physical
$-1/|x|$ (the image of the hole left behind) while the LDA follows the
density's exponential decay. A potential too shallow at the frontier is
precisely why LDA eigenvalues sit too high (Exercises 4 and 5) and why gaps
come out too small ([§8.8](exact-conditions-band-gap.ipynb)).

**Part a)** Recover the exact $v_{xc}$ by rerunning the
[§8.6](hohenberg-kohn.ipynb) inversion on the exact density (the update rule
restated in Setup terms, $\alpha = 0.2$ to $10^{-9}$; exchange-tail gauge as
there), and evaluate the Exercise 5 LDA potential on the same exact density.

**Part b)** Plot both, and quantify the three verdicts by `numpy` arithmetic:
the *energy* verdict ($E_x^{\mathrm{LDA}} = \int n\,\varepsilon_x(n)$ by
`numpy.trapezoid` against the exact $E_x = -J/2$, within ten percent), the
*potential* verdict (the relative gap at the center, near $40\%$), and the
*tail* verdict (at $x = 5$ the exact potential exceeds the LDA's by more than
a factor of 20).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6 — energies forgive, potentials punish

The exchange *energy* must land within $12\%$ of the exact $-J/2$; the
*potential* must miss by $30$–$45\%$ at the center (measured $38\%$); and the
$x = 5$ magnitude ratio must exceed 20 (measured $41$): one functional, three
report cards, all gated.

In [ ]:
validate.check(
    energy_gap < 0.12,
    "the LDA exchange energy lands within 12% of exact",
    f"{100 * energy_gap:.1f}% (energies forgive)",
)
validate.check(
    0.30 < center_gap < 0.45,
    "the LDA potential misses by ~40% at the center",
    f"{100 * center_gap:.1f}% (potentials punish)",
)
validate.check(
    tail_ratio > 20.0,
    "the LDA tail dies too fast by over 20x at x = 5",
    f"exact/LDA magnitude ratio {tail_ratio:.1f}",
)

:::{admonition} With your assistant
:class: tip
The engine's occupations dict makes ions one edit away. Have your assistant
compute the LDA first ionization energy of neon by total-energy differences
($I = E(\mathrm{Ne}^+) - E(\mathrm{Ne})$ with occupations
`{0: [2, 2], 1: [5]}`), then run the check that is yours alone: the
$\Delta$SCF value must land within $15\%$ of the measured $21.56$ eV — far
better than the $-\varepsilon_{2p} = 13.5$ eV Koopmans-style reading, because
total-energy *differences* let the self-interaction error largely cancel
(`numpy` arithmetic, the [§8.1](many-electron-problem.ipynb) eV conversion).
The check is yours.
:::

## Notebook summary

The construction is built and its books are open. The LDA potential was
assembled from the [§8.4](hartree-fock-gas.ipynb) deliverable and certified
against its defining derivative at $10^{-6}$ across five decades of density.
The self-consistent loop behaved as the damped fixed point of
[§0.2](../00-foundations/root-finding.ipynb): one destination for every
damping, iterations falling with $\alpha$. The engine hit the standard
references — He $-2.8343$, Be $-14.4456$, Ne $-128.20$ Ha (totals to
$5\times10^{-4}$, eigenvalues including neon's $2p$ to $3\times10^{-3}$) —
and beryllium's two shells returned with the orbital kinetic energy. Then the
ledger: hydrogen's self-interaction disaster ($-0.446$ Ha, eigenvalue
$-0.234$: half of $-1/2$); a from-scratch LDA for the laboratory's own
interaction (two derivations of its gas exchange agreeing below $10^{-4}$)
landing at $-2.158$ Ha on the exact ladder and missing the ionization theorem
by $38\%$ where the exact functional missed by $0.3\%$; and the face-off's
three report cards — the LDA exchange *energy* within $10\%$ of the exact
$-J/2$, the *potential* off by $38\%$ at the center, and the tail too shallow
by a factor of $41$ at $x = 5$: energies forgive, potentials punish.

## Outlook

- The frontier failures (eigenvalues high, tails shallow) are not random:
  they trace to exact constraints the LDA violates — piecewise linearity in
  particle number above all. [§8.8](exact-conditions-band-gap.ipynb) computes
  those constraints on the laboratory and assembles DFT's deepest practical
  issue, the band gap, from them.
- Everything beyond LDA — gradient corrections, meta-GGAs, hybrids mixing in
  the exact exchange of [§8.3](hartree-fock-atoms.ipynb) — is an attempt to
  keep the sum rule's dividends while fixing the tails and the
  self-interaction; Martin {cite}`martin2004`, Ch. 8, surveys the ladder.
- The engine built here is two steps from a plane-wave solid-state code:
  swap the radial grid for the plane waves of
  [§8.10](plane-waves-pseudopotentials.ipynb) and the nucleus for a
  pseudopotential. The companion MMM course drives exactly such codes; the
  machinery is no longer a black box.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()